In [13]:
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
from dataclasses import dataclass
from langchain.embeddings import init_embeddings
from langgraph.store.memory import InMemoryStore
import uuid
from datetime import datetime

# 사용자 컨텍스트 정의
@dataclass
class Context:
    user_id: str

# 스토어 및 임베딩 초기화
embeddings = init_embeddings("google_genai:gemini-embedding-001")
store = InMemoryStore(
    index={
        "embed": embeddings,
        "dims": 1536,
    }
)

In [15]:
# 초기 데이터 저장
store.put(
    ("users",), 
    "user_123", 
    {
        "user_name": "김일남",
        "user_age": 99,
    }
)

In [16]:
from langchain.tools import ToolRuntime, tool

@tool
def get_user_info(runtime: ToolRuntime[Context]) -> str:
    """사용자의 기본 정보를 조회합니다."""
    assert runtime.store is not None
    user_id = runtime.context.user_id

    user_info = runtime.store.get(("users",), user_id)
    return str(user_info.value) if user_info else "알 수 없는 사용자"

@tool
def save_user_info(
    preferences: list[str] = None,
    interests: list[str] = None,
    experiences: list[str] = None,
    current_activities: list[str] = None,
    goals: list[str] = None,
    routines: list[str] = None,
    concerns: list[str] = None,
    achievements: list[str] = None,
    runtime: ToolRuntime[Context] = None
) -> str:
    """사용자의 다양한 정보를 카테고리별 컬렉션에 저장합니다."""
    assert runtime.store is not None
    store = runtime.store
    user_id = runtime.context.user_id
    current_time = datetime.now().isoformat()

    categories = {
        "preferences": preferences,
        "interests": interests,
        "experiences": experiences,
        "current_activities": current_activities,
        "goals": goals,
        "routines": routines,
        "concerns": concerns,
        "achievements": achievements
    }

    update_summary = []
    for category, values in categories.items():
        if values:
            for value in values:
                item_id = str(uuid.uuid4())
                store.put(
                    (user_id, category),
                    item_id,
                    {
                        "text": value,
                        "created_at": current_time,
                        "category": category
                    }
                )
            update_summary.append(f"{len(values)}개의 {category}")

    return f"사용자 메모리가 성공적으로 저장되었습니다: {', '.join(update_summary)}가 추가되었습니다."

@tool
def search_user_memories(
    query: str,
    category: str = None,
    limit: int = 5,
    runtime: ToolRuntime[Context] = None
) -> str:
    """사용자의 메모리를 검색합니다."""
    assert runtime.store is not None
    store = runtime.store
    user_id = runtime.context.user_id

    if category:
        namespace = (user_id, category)
        results = store.search(namespace, query=query, limit=limit)
        if not results:
            return f"{category} 카테고리에서 관련된 메모리를 찾을 수 없습니다."

        result_text = f"{category} 관련 메모리:\n"
        for item in results:
            result_text += f"- {item.value['text']} (저장일자: {item.value['created_at']})\n"
        return result_text

    else:
        categories = ["preferences", "interests", "experiences", "current_activities",
                     "goals", "routines", "concerns", "achievements"]
        all_results = []
        for cat in categories:
            try:
                results = store.search((user_id, cat), query=query, limit=limit)
                all_results.extend([(r, cat) for r in results])
            except:
                continue

        all_results.sort(key=lambda x: x[0].score if hasattr(x[0], 'score') else 0, reverse=True)
        all_results = all_results[:limit]

        if not all_results:
            return "관련된 메모리를 찾을 수 없습니다."

        result_text = "관련 메모리:\n"
        for item, cat in all_results:
            result_text += f"[{cat}] {item.value['text']} (저장일자: {item.value['created_at']})\n"
        return result_text

In [17]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-3.1-flash-lite")
checkpointer = InMemorySaver()

# 에이전트 생성 시 context_schema 전달
agent = create_agent(
    model=model,
    tools=[get_user_info, save_user_info, search_user_memories],
    store=store,
    context_schema=Context,  # 스키마 등록
    system_prompt="""당신은 누적된 사용자 메모리를 활용하여 맞춤 조언을 제공하는 친절한 라이프 코치 어시스턴트입니다... (기존 시스템 프롬프트와 동일)""",
    checkpointer=checkpointer,
)

In [18]:
# 실행 루프
while True:
    user_input = input("User: ")
    if user_input.lower() in ["q", "exit", "quit"]:
        break

    # invoke 시 context 매개변수를 통해 데이터 전달
    response = agent.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config={"configurable": {"thread_id": "1"}},
        context=Context(user_id="user_123")  # 사용자 식별자 주입
    )

    for msg in response["messages"]:
        msg.pretty_print()

================================ Human Message =================================

내 이름이 뭐야?
================================== Ai Message ==================================

[]
Tool Calls:
  get_user_info (8c7b108d-e521-43b3-9758-f907c8373211)
 Call ID: 8c7b108d-e521-43b3-9758-f907c8373211
  Args:
================================= Tool Message =================================
Name: get_user_info

{'user_name': '김일남', 'user_age': 99}
================================== Ai Message ==================================

[{'type': 'text', 'text': '사용자님의 성함은 **김일남** 님입니다! 무엇을 도와드릴까요?', 'extras': {'signature': 'EjQKMgEMOdbHxQAYLdrCL7okU+rVqXGUEdPfYcaXaWyZx4rSAixIzeIyczpTkplhGANUJzEC'}}]
================================ Human Message =================================

내 이름이 뭐야?
================================== Ai Message ==================================

[]
Tool Calls:
  get_user_info (8c7b108d-e521-43b3-9758-f907c8373211)
 Call ID: 8c7b108d-e521-43b3-9758-f907c8373211
  Args:
==============